# Birr Track — Fine-tune Qwen2.5-VL-3B (Kaggle)

Pip-safe notebook for LoRA SFT on receipt screenshots via [LLaMA-Factory](https://github.com/hiyouga/LLaMA-Factory) v0.9.2.

## Kaggle settings (required)

1. **Language → Python 3.10** (not 3.12 — avoids `pkgutil.ImpImporter` crashes).
2. **Accelerator → GPU T4 x2**.
3. **Internet → ON** (Hugging Face model download).
4. **Add Data** → attach dataset `birrtrack-receipts` (see repo `package_kaggle_dataset.py`).
5. Run all cells. Download `qwen25vl-3b-birrtrack-lora.zip` from Output.


In [ ]:
import subprocess
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"]).decode())
print(subprocess.check_output(["python", "--version"]).decode())


## 1. Install (minimal — do not override peft/numpy unless the next cell fails)

In [ ]:
!pip install -q --upgrade pip wheel
!pip install -q --upgrade "setuptools>=70"
!rm -rf /kaggle/working/LLaMA-Factory
!git clone --depth 1 --branch v0.9.2 https://github.com/hiyouga/LLaMA-Factory.git /kaggle/working/LLaMA-Factory
!pip install -q "/kaggle/working/LLaMA-Factory[torch,metrics]"
!pip install -q "transformers==4.49.0" qwen-vl-utils==0.0.8

import importlib
import importlib.metadata

for _name in ("torch", "transformers", "peft"):
    _m = importlib.import_module(_name)
    print(_name, getattr(_m, "__version__", "?"))
print("llamafactory", importlib.metadata.version("llamafactory"))

import transformers as _tf
assert _tf.__version__ >= "4.49.0", (
    f"Need transformers>=4.49.0, got {_tf.__version__}. Set Language to Python 3.10 and re-run."
)
from transformers import Qwen2_5_VLForConditionalGeneration  # noqa: F401

print("Install OK — Qwen2.5-VL classes import successfully.")



## 2. Dataset paths (rewrite image paths for Kaggle mount)

In [ ]:
import json
import os
from pathlib import Path

KAGGLE_DATASET_NAME = "birrtrack-receipts"
KAGGLE_DATASET_ROOT = Path(f"/kaggle/input/{KAGGLE_DATASET_NAME}")

if not KAGGLE_DATASET_ROOT.is_dir():
    candidates = sorted(Path("/kaggle/input").glob("*"))
    raise FileNotFoundError(
        f"Dataset not found at {KAGGLE_DATASET_ROOT}. "
        f"Available: {[c.name for c in candidates]}"
    )

receipts_dir = KAGGLE_DATASET_ROOT / "receipts"
train_src = KAGGLE_DATASET_ROOT / "train.json"
val_src = KAGGLE_DATASET_ROOT / "val.json"

for required in (receipts_dir, train_src, val_src):
    if not required.exists():
        raise FileNotFoundError(f"Missing in Kaggle dataset: {required}")

work_data_dir = Path("/kaggle/working/birrtrack_data")
work_data_dir.mkdir(parents=True, exist_ok=True)


def rewrite_image_paths(src_json: Path, dst_json: Path) -> tuple[int, int]:
    examples = json.loads(src_json.read_text(encoding="utf-8"))
    rewritten = 0
    missing: list[str] = []
    for ex in examples:
        new_images = []
        for img in ex.get("images", []):
            file_name = os.path.basename(img)
            new_path = receipts_dir / file_name
            if not new_path.is_file():
                missing.append(file_name)
            new_images.append(str(new_path))
            rewritten += 1
        ex["images"] = new_images
    if missing:
        raise FileNotFoundError(
            f"{len(missing)} image(s) missing from {receipts_dir}. First: {missing[:5]}"
        )
    dst_json.write_text(json.dumps(examples, ensure_ascii=False, indent=2), encoding="utf-8")
    return len(examples), rewritten


train_dst = work_data_dir / "train.json"
val_dst = work_data_dir / "val.json"
train_count, _ = rewrite_image_paths(train_src, train_dst)
val_count, _ = rewrite_image_paths(val_src, val_dst)
print(f"Train: {train_count} examples | Val: {val_count} examples")



## 3. Register dataset with LLaMA-Factory

In [ ]:
dataset_info = {
    "birrtrack_train": {
        "file_name": str(train_dst),
        "formatting": "sharegpt",
        "columns": {"messages": "conversations", "images": "images"},
        "tags": {
            "role_tag": "from",
            "content_tag": "value",
            "user_tag": "human",
            "assistant_tag": "gpt",
        },
    },
    "birrtrack_val": {
        "file_name": str(val_dst),
        "formatting": "sharegpt",
        "columns": {"messages": "conversations", "images": "images"},
        "tags": {
            "role_tag": "from",
            "content_tag": "value",
            "user_tag": "human",
            "assistant_tag": "gpt",
        },
    },
}
dataset_info_path = work_data_dir / "dataset_info.json"
dataset_info_path.write_text(json.dumps(dataset_info, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote {dataset_info_path}")



## 4. Training config (matches `write_train_config.py` in the repo)

In [ ]:
# Same hyperparameters as qwen-vlm-training/write_train_config.py base_train_config (CUDA fp16).

OUTPUT_DIR = Path("/kaggle/working/qwen25vl-3b-birrtrack-lora")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_config_lines = [
    'model_name_or_path: "Qwen/Qwen2.5-VL-3B-Instruct"',
    "trust_remote_code: true",
    'stage: "sft"',
    "do_train: true",
    'finetuning_type: "lora"',
    'lora_target: "all"',
    "lora_rank: 16",
    "lora_alpha: 32",
    "lora_dropout: 0.05",
    "freeze_vision_tower: true",
    'dataset: "birrtrack_train"',
    'eval_dataset: "birrtrack_val"',
    f'dataset_dir: "{work_data_dir}"',
    'template: "qwen2_vl"',
    "cutoff_len: 2048",
    "overwrite_cache: true",
    "preprocessing_num_workers: 2",
    f'output_dir: "{OUTPUT_DIR}"',
    "overwrite_output_dir: true",
    "logging_steps: 5",
    "save_steps: 50",
    "save_total_limit: 2",
    "plot_loss: true",
    "per_device_train_batch_size: 1",
    "per_device_eval_batch_size: 1",
    "gradient_accumulation_steps: 8",
    "learning_rate: 1e-4",
    "num_train_epochs: 10",
    'lr_scheduler_type: "cosine"',
    "warmup_ratio: 0.03",
    "fp16: true",
    "bf16: false",
    "gradient_checkpointing: true",
    'eval_strategy: "epoch"',
    "load_best_model_at_end: false",
    'report_to: "none"',
    "seed: 42",
]
config_path = Path("/kaggle/working/train_config.yaml")
config_path.write_text("\n".join(train_config_lines) + "\n", encoding="utf-8")
print(config_path.read_text())



## 5. Train

In [ ]:
import os
import subprocess
import sys
import sysconfig

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall", "setuptools>=70"]
)
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["WANDB_DISABLED"] = "true"
env["WANDB_MODE"] = "disabled"
env["HF_HUB_DISABLE_TELEMETRY"] = "1"
env["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
env.setdefault("TOKENIZERS_PARALLELISM", "false")
env.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
_platlib = sysconfig.get_path("platlib")
if _platlib:
    _pp = env.get("PYTHONPATH", "").strip()
    env["PYTHONPATH"] = _platlib + ((os.pathsep + _pp) if _pp else "")

subprocess.check_call(
    [sys.executable, "-m", "llamafactory.cli", "train", str(config_path)],
    env=env,
)

_adapter_cfg = OUTPUT_DIR / "adapter_config.json"
_adapter_weights = OUTPUT_DIR / "adapter_model.safetensors"
if not _adapter_cfg.is_file() or not _adapter_weights.is_file():
    raise FileNotFoundError(
        f"No adapter in {OUTPUT_DIR}. Contents: {sorted(p.name for p in OUTPUT_DIR.iterdir())}"
    )
print(f"Adapter saved: {OUTPUT_DIR}")



## 6. Sanity-check inference (one validation image)

In [ ]:
import gc

import torch
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

BASE_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map={"": DEVICE},
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))
model.eval()
pad_token_id = processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id

val_examples = json.loads(val_dst.read_text(encoding="utf-8"))
sample = val_examples[0]
sample_image_path = sample["images"][0]
sample_label = sample["conversations"][1]["value"]
user_text = sample["conversations"][0]["value"].replace("<image>", "").strip()

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": sample_image_path},
            {"type": "text", "text": user_text},
        ],
    }
]
text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text_prompt],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
).to(DEVICE)

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=pad_token_id,
    )

input_lengths = inputs.input_ids.shape[1]
generated = processor.batch_decode(output_ids[:, input_lengths:].cpu(), skip_special_tokens=True)[0]
print(f"Image:    {sample_image_path}")
print(f"Expected: {sample_label}")
print(f"Got:      {generated.strip()}")



## 7. Zip adapter for download

In [ ]:
import os
import shutil

zip_path = shutil.make_archive(
    base_name="/kaggle/working/qwen25vl-3b-birrtrack-lora",
    format="zip",
    root_dir=str(OUTPUT_DIR),
)
print(f"Download from Output: {zip_path} ({os.path.getsize(zip_path) / 1024 / 1024:.1f} MB)")

